In [ ]:
from importlib.resources import as_file, files
import os
from pathlib import Path

from openghg_inversions.rhime import run_rhime

tutorial_output_path = Path(os.environ.get("OPENGHG_TUTORIAL_OUTPUT_PATH", "outputs"))
resource = files("openghg_inversions.rhime").joinpath("config/standard_tutorial.ini")
with as_file(resource) as config:
    result = run_rhime(config_file=config, output_path=tutorial_output_path)

{
    "OpenGHG Inversions commit": os.environ.get(
        "OPENGHG_TUTORIAL_CODE_REF", "local checkout"
    ),
    "tutorial data": os.environ.get("OPENGHG_TUTORIAL_DATA_TAG", "v1.0.0"),
    "sites": list(result.run_spec.sites),
    "observations": result.inv_inputs.sizes["nmeasure"],
    "posterior samples": {
        name: result.idata.posterior.sizes[name] for name in ("chain", "draw")
    },
}

{'OpenGHG Inversions commit': '9202222950c3ef2eb7bc83d040df3bcf6c26f94c',
 'tutorial data': 'v1.0.0',
 'sites': ['MHD', 'TAC'],
 'observations': 84,
 'posterior samples': {'chain': 2, 'draw': 50}}

In [ ]:
measurement_index = result.inv_inputs.indexes["nmeasure"]
{
    "period": (result.run_spec.start_date, result.run_spec.end_date),
    "sectors": [
        (sector.name, sector.flux_source) for sector in result.model_spec.sectors
    ],
    "H dimensions": result.inv_inputs["H"].dims,
    "input sizes": dict(result.inv_inputs.sizes),
    "measurement sites": measurement_index.get_level_values("site").unique().tolist(),
    "x dimensions": result.idata.posterior["x"].dims,
    "variable roles": result.model_build_result.variable_roles,
    "output products": sorted(result.outputs),
}

{'period': ('2020-01-01', '2020-01-08'),
 'sectors': [('edgar-v80-anthropogenic', 'edgar-v80-anthropogenic')],
 'H dimensions': ('region', 'nmeasure'),
 'input sizes': {'nmeasure': 84,
  'lat': 293,
  'lon': 391,
  'height': 20,
  'region': 4,
  'bc_region': 4,
  'nsite': 2},
 'measurement sites': ['MHD', 'TAC'],
 'x dimensions': ('chain', 'draw', 'region'),
 'variable roles': {'observation': 'mf',
  'observation_error': 'mf_error',
  'minimum_error': 'min_error',
  'concentration': 'y',
  'model_error': 'epsilon',
  'observation_repeatability': 'mf_repeatability',
  'observation_variability': 'mf_variability',
  'flux_scale': 'x',
  'flux_contribution': 'mu',
  'emissions_sensitivity': 'hx',
  'boundary': 'mu_bc',
  'baseline_scale': 'bc',
  'baseline_sensitivity': 'hbc',
  'baseline': 'mu_bc'},
 'output products': ['inversion_output']}

In [ ]:
import arviz as az

summary = az.summary(
    result.idata,
    var_names=["x", "bc", "sigma"],
    kind="diagnostics",
)
summary["divergences"] = int(result.idata.sample_stats["diverging"].sum())
summary.round(2)

              mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat  divergences
x[0]               0.03     0.04      12.0      23.0   1.36            0
x[1]               0.22     0.02       3.0      23.0   2.36            0
x[2]               0.38     0.05       3.0      23.0   2.00            0
x[3]               0.07     0.03       5.0      18.0   1.45            0
bc[('n', 0)]       0.19     0.12       4.0      13.0   1.80            0
bc[('e', 0)]       0.01     0.00      18.0      49.0   1.36            0
bc[('s', 0)]       0.14     0.03       4.0      19.0   1.68            0
bc[('w', 0)]       0.00     0.00       3.0      17.0   1.81            0
sigma[0, 0]        0.21     0.03       3.0      18.0   2.44            0
sigma[1, 0]        0.11     0.09       4.0      13.0   1.52            0

In [ ]:
from openghg_inversions.postprocessing.inversion_output import InversionOutput

inv_out = result.outputs["inversion_output"]
saved = Path(result.output_metadata["inversion_output_path"])
reloaded = InversionOutput.load(saved)
{
    "provenance contract": reloaded.provenance["contract"],
    "basis artifact source": reloaded.run_metadata["basis_artifact_source"],
    "split by sectors": reloaded.run_metadata["split_by_sectors"],
    "saved file": saved.name,
    "posterior variables": sorted(reloaded.trace.posterior.data_vars),
    "posterior sizes": dict(reloaded.trace.posterior.sizes),
    "sampler fields": sorted(reloaded.output_metadata["sampler"]),
}

{'provenance contract': 'modern_rhime_inversion_output',
 'basis artifact source': 'generated',
 'split by sectors': False,
 'saved file': 'standard_tutorial2020-01-01_inversion_output.nc',
 'posterior variables': ['bc', 'epsilon', 'mu', 'mu_bc', 'sigma', 'x'],
 'posterior sizes': {'chain': 2,
  'draw': 50,
  'region': 4,
  'bc_region': 4,
  'nsigma_site': 2,
  'nsigma_time': 1,
  'nmeasure': 84},
 'sampler fields': ['burn', 'chains', 'draws', 'nuts_sampler', 'tune']}